# Customer Churn Prediction

End-to-end ML project: EDA, feature engineering, model training, and evaluation on the Telco Customer Churn dataset.

**Goal:** Build the most accurate model to predict whether a customer will churn.

## 1. Import Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import pickle
import warnings

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    VotingClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, f1_score, roc_curve, precision_recall_curve,
)
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

warnings.filterwarnings("ignore")
plt.style.use('fivethirtyeight')
plt.rcParams['figure.figsize'] = (12, 5)
np.random.seed(42)

print("Libraries loaded successfully!")

## 2. Load & Explore Data

In [ ]:
df = pd.read_csv("dataset_telco.csv")
print(f"Dataset shape: {df.shape}")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().T

In [ ]:
# Check for missing values
print("Missing values per column:")
print(df.isnull().sum())
print(f"\nTotal missing: {df.isnull().sum().sum()}")

In [ ]:
# Check TotalCharges - has spaces instead of NaN
print(f"Empty strings in TotalCharges: {len(df[df['TotalCharges'] == ' '])}")
df.loc[df['TotalCharges'] == ' '][['tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']]

## 3. Data Cleaning

In [ ]:
# Drop customerID - not a predictive feature
df.drop(columns=['customerID'], inplace=True)

# Fix TotalCharges: replace spaces with NaN, convert to numeric
df['TotalCharges'] = df['TotalCharges'].replace(' ', np.nan)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# For customers with tenure=0, fill TotalCharges with MonthlyCharges
mask = df['TotalCharges'].isna()
df.loc[mask, 'TotalCharges'] = df.loc[mask, 'MonthlyCharges']
print(f"Fixed {mask.sum()} missing TotalCharges values")

# Encode target variable
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
print(f"\nChurn distribution:")
print(df['Churn'].value_counts())
print(f"\nChurn rate: {df['Churn'].mean()*100:.1f}%")

## 4. Exploratory Data Analysis (EDA)

### 4.1 Target Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
churn_counts = df['Churn'].value_counts()
colors = ['#2ecc71', '#e74c3c']
axes[0].bar(['No Churn', 'Churn'], churn_counts.values, color=colors, edgecolor='black', alpha=0.8)
axes[0].set_title('Churn Distribution (Count)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 50, str(v), ha='center', fontweight='bold', fontsize=12)

# Pie chart
axes[1].pie(churn_counts.values, labels=['No Churn', 'Churn'], colors=colors,
           autopct='%1.1f%%', startangle=90, textprops={'fontsize': 12})
axes[1].set_title('Churn Distribution (%)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"Class imbalance ratio: {churn_counts[0]/churn_counts[1]:.2f}:1")

### 4.2 Numerical Feature Distributions

In [ ]:
def plot_distribution(df, column_name):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Histogram by churn
    for label, color in [(0, '#2ecc71'), (1, '#e74c3c')]:
        subset = df[df['Churn'] == label][column_name]
        axes[0].hist(subset, bins=30, alpha=0.6, color=color,
                    label='No Churn' if label == 0 else 'Churn', edgecolor='black')
    axes[0].set_title(f'{column_name} Distribution by Churn', fontweight='bold')
    axes[0].legend()

    # KDE plot
    sns.kdeplot(data=df, x=column_name, hue='Churn', ax=axes[1], fill=True, alpha=0.4,
               palette={0: '#2ecc71', 1: '#e74c3c'})
    axes[1].set_title(f'{column_name} Density by Churn', fontweight='bold')

    # Boxplot
    sns.boxplot(data=df, x='Churn', y=column_name, ax=axes[2],
               palette={0: '#2ecc71', 1: '#e74c3c'})
    axes[2].set_title(f'{column_name} Boxplot by Churn', fontweight='bold')
    axes[2].set_xticklabels(['No Churn', 'Churn'])

    plt.tight_layout()
    plt.show()

for col in ['tenure', 'MonthlyCharges', 'TotalCharges']:
    plot_distribution(df, col)

### 4.3 Correlation Heatmap

In [ ]:
plt.figure(figsize=(10, 6))
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr = df[numeric_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, cmap='RdBu_r', fmt='.2f', mask=mask,
           center=0, square=True, linewidths=1)
plt.title('Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 4.4 Categorical Features vs Churn

In [ ]:
categorical_cols = df.select_dtypes(include='object').columns.tolist() + ['SeniorCitizen']

fig, axes = plt.subplots(4, 4, figsize=(24, 20))
axes = axes.flatten()

for i, col in enumerate(categorical_cols):
    if i < len(axes):
        ct = pd.crosstab(df[col], df['Churn'], normalize='index') * 100
        ct.plot(kind='bar', stacked=True, ax=axes[i], color=['#2ecc71', '#e74c3c'],
               edgecolor='black', alpha=0.8)
        axes[i].set_title(f'{col}', fontweight='bold', fontsize=11)
        axes[i].set_ylabel('Percentage')
        axes[i].legend(['No Churn', 'Churn'], fontsize=8)
        axes[i].tick_params(axis='x', rotation=45)

# Hide unused subplots
for j in range(len(categorical_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Churn Rate by Categorical Features', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 4.5 Key Business Insights

In [ ]:
# Churn rate by contract type
print("=== Churn Rate by Contract Type ===")
print(df.groupby('Contract')['Churn'].mean().sort_values(ascending=False).apply(lambda x: f"{x*100:.1f}%"))

print("\n=== Churn Rate by Internet Service ===")
print(df.groupby('InternetService')['Churn'].mean().sort_values(ascending=False).apply(lambda x: f"{x*100:.1f}%"))

print("\n=== Churn Rate by Payment Method ===")
print(df.groupby('PaymentMethod')['Churn'].mean().sort_values(ascending=False).apply(lambda x: f"{x*100:.1f}%"))

print("\n=== Churn Rate by Senior Citizen ===")
print(df.groupby('SeniorCitizen')['Churn'].mean().apply(lambda x: f"{x*100:.1f}%"))

## 5. Feature Engineering

In [ ]:
# Average charges per month
df['AvgChargesPerMonth'] = df['TotalCharges'] / (df['tenure'] + 1)

# Tenure groups (lifecycle stage)
df['TenureGroup'] = pd.cut(
    df['tenure'], bins=[-1, 12, 24, 48, 60, 72],
    labels=['0-12', '13-24', '25-48', '49-60', '61-72']
)

# Charges ratio
df['ChargesRatio'] = df['TotalCharges'] / (df['MonthlyCharges'] * (df['tenure'] + 1) + 1)

# Number of services
service_cols = ['PhoneService', 'MultipleLines', 'InternetService',
                'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                'TechSupport', 'StreamingTV', 'StreamingMovies']
df['NumServices'] = sum(
    (df[c].isin(['Yes', 'DSL', 'Fiber optic'])).astype(int) for c in service_cols
)

# Has internet
df['HasInternet'] = (df['InternetService'] != 'No').astype(int)

# Has bundle (phone + internet)
df['HasBundle'] = ((df['PhoneService'] == 'Yes') & (df['InternetService'] != 'No')).astype(int)

# Charges per service
df['ChargesPerService'] = df['MonthlyCharges'] / (df['NumServices'] + 1)

# New customer flag
df['IsNewCustomer'] = (df['tenure'] <= 12).astype(int)

# Simplify redundant labels
df['MultipleLines'] = df['MultipleLines'].replace('No phone service', 'No')
for col in ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
            'TechSupport', 'StreamingTV', 'StreamingMovies']:
    df[col] = df[col].replace('No internet service', 'No')

# Security features count
df['NumSecurityFeatures'] = sum(
    (df[c] == 'Yes').astype(int) for c in
    ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport']
)

# Streaming features count
df['NumStreamingFeatures'] = sum(
    (df[c] == 'Yes').astype(int) for c in ['StreamingTV', 'StreamingMovies']
)

print(f"Total features after engineering: {df.shape[1] - 1}")
print(f"\nNew features: AvgChargesPerMonth, TenureGroup, ChargesRatio, NumServices,")
print(f"HasInternet, HasBundle, ChargesPerService, IsNewCustomer, NumSecurityFeatures, NumStreamingFeatures")
df.head()

In [ ]:
# Visualize engineered features
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Churn by tenure group
ct = pd.crosstab(df['TenureGroup'], df['Churn'], normalize='index') * 100
ct.plot(kind='bar', stacked=True, ax=axes[0, 0], color=['#2ecc71', '#e74c3c'])
axes[0, 0].set_title('Churn Rate by Tenure Group', fontweight='bold')
axes[0, 0].set_ylabel('Percentage')

# Churn by number of services
ct2 = pd.crosstab(df['NumServices'], df['Churn'], normalize='index') * 100
ct2.plot(kind='bar', stacked=True, ax=axes[0, 1], color=['#2ecc71', '#e74c3c'])
axes[0, 1].set_title('Churn Rate by Number of Services', fontweight='bold')

# Avg charges per month by churn
sns.boxplot(data=df, x='Churn', y='AvgChargesPerMonth', ax=axes[0, 2],
           palette={0: '#2ecc71', 1: '#e74c3c'})
axes[0, 2].set_title('Avg Charges/Month by Churn', fontweight='bold')
axes[0, 2].set_xticklabels(['No Churn', 'Churn'])

# Security features
ct3 = pd.crosstab(df['NumSecurityFeatures'], df['Churn'], normalize='index') * 100
ct3.plot(kind='bar', stacked=True, ax=axes[1, 0], color=['#2ecc71', '#e74c3c'])
axes[1, 0].set_title('Churn by Security Features Count', fontweight='bold')

# Bundle effect
ct4 = pd.crosstab(df['HasBundle'], df['Churn'], normalize='index') * 100
ct4.plot(kind='bar', stacked=True, ax=axes[1, 1], color=['#2ecc71', '#e74c3c'])
axes[1, 1].set_title('Churn Rate: Bundle vs No Bundle', fontweight='bold')
axes[1, 1].set_xticklabels(['No Bundle', 'Has Bundle'])

# New customer effect
ct5 = pd.crosstab(df['IsNewCustomer'], df['Churn'], normalize='index') * 100
ct5.plot(kind='bar', stacked=True, ax=axes[1, 2], color=['#2ecc71', '#e74c3c'])
axes[1, 2].set_title('Churn: New vs Established Customers', fontweight='bold')
axes[1, 2].set_xticklabels(['Established', 'New (<12mo)'])

plt.tight_layout()
plt.show()

## 6. Preprocessing Pipeline

In [ ]:
categorical_features = [
    'gender', 'Partner', 'Dependents', 'PhoneService',
    'MultipleLines', 'InternetService', 'OnlineSecurity',
    'OnlineBackup', 'DeviceProtection', 'TechSupport',
    'StreamingTV', 'StreamingMovies', 'Contract',
    'PaperlessBilling', 'PaymentMethod', 'TenureGroup',
]

numerical_features = [
    'SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges',
    'AvgChargesPerMonth', 'ChargesRatio', 'NumServices',
    'HasInternet', 'HasBundle', 'ChargesPerService',
    'IsNewCustomer', 'NumSecurityFeatures', 'NumStreamingFeatures',
]

print(f"Categorical features: {len(categorical_features)}")
print(f"Numerical features:   {len(numerical_features)}")

# No data leakage: fit only on training data!
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numerical_features),
    ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), categorical_features),
], remainder='drop')

X = df.drop(columns=['Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_tr = preprocessor.fit_transform(X_train)
X_te = preprocessor.transform(X_test)

print(f"\nTrain shape: {X_tr.shape}")
print(f"Test shape:  {X_te.shape}")
print(f"\nTrain class distribution: {dict(y_train.value_counts())}")

### 6.1 Handle Class Imbalance with SMOTE

In [ ]:
smote = SMOTE(random_state=42)
X_sm, y_sm = smote.fit_resample(X_tr, y_train)

print(f"Before SMOTE: {X_tr.shape[0]} samples")
print(f"After SMOTE:  {X_sm.shape[0]} samples")
print(f"\nClass distribution after SMOTE:")
print(pd.Series(y_sm).value_counts())

## 7. Model Training & Comparison

### 7.1 Baseline Comparison

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

baselines = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=200, eval_metric='logloss', random_state=42, verbosity=0),
}

print("5-Fold Cross-Validation (SMOTE Training Data)")
print("=" * 55)
results = {}
for name, model in baselines.items():
    scores = cross_val_score(model, X_sm, y_sm, cv=cv, scoring='f1', n_jobs=-1)
    results[name] = scores
    print(f"{name:25s} | F1: {scores.mean():.4f} (+/- {scores.std():.4f})")

# Visualize
fig, ax = plt.subplots(figsize=(10, 5))
bp = ax.boxplot([results[name] for name in results], labels=results.keys(), patch_artist=True)
colors = ['#3498db', '#2ecc71', '#e67e22', '#e74c3c']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_title('Model Comparison (5-fold CV F1 Score)', fontweight='bold', fontsize=14)
ax.set_ylabel('F1 Score')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

### 7.2 Hyperparameter Tuning

In [ ]:
# XGBoost + SMOTE
print("Tuning XGBoost + SMOTE...")
xgb_grid = GridSearchCV(
    XGBClassifier(eval_metric='logloss', random_state=42, verbosity=0),
    {'n_estimators': [200, 300], 'max_depth': [5, 7], 'learning_rate': [0.05, 0.1],
     'min_child_weight': [1, 3], 'subsample': [0.8], 'colsample_bytree': [0.8, 1.0]},
    cv=cv, scoring='f1', n_jobs=-1,
).fit(X_sm, y_sm)
print(f"Best F1: {xgb_grid.best_score_:.4f}")
print(f"Params:  {xgb_grid.best_params_}")

In [ ]:
# GradientBoosting + SMOTE
print("Tuning GradientBoosting + SMOTE...")
gb_grid = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    {'n_estimators': [200, 300], 'max_depth': [3, 5], 'learning_rate': [0.05, 0.1], 'subsample': [0.8]},
    cv=cv, scoring='f1', n_jobs=-1,
).fit(X_sm, y_sm)
print(f"Best F1: {gb_grid.best_score_:.4f}")
print(f"Params:  {gb_grid.best_params_}")

In [ ]:
# RandomForest + SMOTE
print("Tuning RandomForest + SMOTE...")
rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    {'n_estimators': [300, 500], 'max_depth': [20, None], 'min_samples_split': [2, 5]},
    cv=cv, scoring='f1', n_jobs=-1,
).fit(X_sm, y_sm)
print(f"Best F1: {rf_grid.best_score_:.4f}")
print(f"Params:  {rf_grid.best_params_}")

In [ ]:
# XGBoost with class weights (no SMOTE)
print("Tuning XGBoost + class_weight (no SMOTE)...")
neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
xgb_cw = GridSearchCV(
    XGBClassifier(eval_metric='logloss', random_state=42, verbosity=0),
    {'n_estimators': [300, 500], 'max_depth': [3, 5, 7], 'learning_rate': [0.01, 0.05, 0.1],
     'scale_pos_weight': [1, neg/pos], 'subsample': [0.8], 'colsample_bytree': [0.8]},
    cv=cv, scoring='roc_auc', n_jobs=-1,
).fit(X_tr, y_train)
print(f"Best AUC: {xgb_cw.best_score_:.4f}")
print(f"Params:   {xgb_cw.best_params_}")

In [ ]:
# GradientBoosting on original
print("Tuning GradientBoosting (original data)...")
gb_orig = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    {'n_estimators': [300, 500], 'max_depth': [3, 5], 'learning_rate': [0.05, 0.1], 'subsample': [0.8, 1.0]},
    cv=cv, scoring='roc_auc', n_jobs=-1,
).fit(X_tr, y_train)
print(f"Best AUC: {gb_orig.best_score_:.4f}")
print(f"Params:   {gb_orig.best_params_}")

## 8. Model Evaluation on Test Set

In [ ]:
def evaluate_with_threshold(model, X_test, y_test, name="Model"):
    """Evaluate model with optimized threshold."""
    y_prob = model.predict_proba(X_test)[:, 1]

    # Find optimal threshold for accuracy
    best_thresh, best_acc = 0.5, 0
    for t in np.arange(0.30, 0.70, 0.005):
        acc = accuracy_score(y_test, (y_prob >= t).astype(int))
        if acc > best_acc:
            best_acc = acc
            best_thresh = t

    y_pred = (y_prob >= best_thresh).astype(int)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc = roc_auc_score(y_test, y_prob)

    return {
        'name': name, 'model': model, 'acc': acc, 'f1': f1,
        'roc': roc, 'thresh': best_thresh, 'y_pred': y_pred, 'y_prob': y_prob,
    }

# Evaluate all models
results = []
for name, est in [
    ('XGB-SMOTE', xgb_grid.best_estimator_),
    ('GB-SMOTE', gb_grid.best_estimator_),
    ('RF-SMOTE', rf_grid.best_estimator_),
    ('XGB-ClassWeight', xgb_cw.best_estimator_),
    ('GB-Original', gb_orig.best_estimator_),
]:
    r = evaluate_with_threshold(est, X_te, y_test, name)
    results.append(r)

# Voting ensembles
vote_smote = VotingClassifier(
    estimators=[('xgb', xgb_grid.best_estimator_), ('gb', gb_grid.best_estimator_), ('rf', rf_grid.best_estimator_)],
    voting='soft', n_jobs=-1,
).fit(X_sm, y_sm)
results.append(evaluate_with_threshold(vote_smote, X_te, y_test, 'Vote-SMOTE'))

vote_cw = VotingClassifier(
    estimators=[('xgb', xgb_cw.best_estimator_), ('gb', gb_orig.best_estimator_)],
    voting='soft', n_jobs=-1,
).fit(X_tr, y_train)
results.append(evaluate_with_threshold(vote_cw, X_te, y_test, 'Vote-CW'))

# Display results
results_df = pd.DataFrame([{
    'Model': r['name'], 'Accuracy': r['acc'], 'F1': r['f1'],
    'ROC-AUC': r['roc'], 'Threshold': r['thresh']
} for r in results]).sort_values('Accuracy', ascending=False)

print("\n" + "="*65)
print("MODEL COMPARISON ON TEST SET")
print("="*65)
print(results_df.to_string(index=False))

In [ ]:
# Visualize model comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Sort by accuracy
sorted_results = sorted(results, key=lambda x: x['acc'], reverse=True)
names = [r['name'] for r in sorted_results]
colors_map = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(names)))

# Accuracy
accs = [r['acc'] for r in sorted_results]
bars = axes[0].barh(names, accs, color=colors_map, edgecolor='black')
axes[0].set_xlabel('Accuracy')
axes[0].set_title('Test Accuracy', fontweight='bold')
axes[0].set_xlim(0.7, 0.85)
for bar, v in zip(bars, accs):
    axes[0].text(v + 0.002, bar.get_y() + bar.get_height()/2, f'{v:.4f}', va='center', fontweight='bold')

# F1
f1s = [r['f1'] for r in sorted_results]
bars = axes[1].barh(names, f1s, color=colors_map, edgecolor='black')
axes[1].set_xlabel('F1 Score')
axes[1].set_title('Test F1 Score', fontweight='bold')
axes[1].set_xlim(0.4, 0.7)
for bar, v in zip(bars, f1s):
    axes[1].text(v + 0.002, bar.get_y() + bar.get_height()/2, f'{v:.4f}', va='center', fontweight='bold')

# ROC-AUC
aucs = [r['roc'] for r in sorted_results]
bars = axes[2].barh(names, aucs, color=colors_map, edgecolor='black')
axes[2].set_xlabel('ROC-AUC')
axes[2].set_title('Test ROC-AUC', fontweight='bold')
axes[2].set_xlim(0.75, 0.9)
for bar, v in zip(bars, aucs):
    axes[2].text(v + 0.002, bar.get_y() + bar.get_height()/2, f'{v:.4f}', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 9. Best Model Analysis

In [ ]:
# Select best model
best = max(results, key=lambda x: (x['acc'], x['roc']))
print(f"Best Model: {best['name']}")
print(f"Accuracy:   {best['acc']:.4f}")
print(f"F1 Score:   {best['f1']:.4f}")
print(f"ROC-AUC:    {best['roc']:.4f}")
print(f"Threshold:  {best['thresh']:.3f}")
print()
print("Classification Report:")
print(classification_report(y_test, best['y_pred'], target_names=['No Churn', 'Churn']))

In [ ]:
# Confusion Matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix heatmap
cm = confusion_matrix(y_test, best['y_pred'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
           xticklabels=['No Churn', 'Churn'], yticklabels=['No Churn', 'Churn'])
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title(f'Confusion Matrix - {best["name"]}', fontweight='bold')

# Normalized
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues', ax=axes[1],
           xticklabels=['No Churn', 'Churn'], yticklabels=['No Churn', 'Churn'])
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')
axes[1].set_title('Normalized Confusion Matrix', fontweight='bold')

plt.tight_layout()
plt.show()

print(f"True Positives:  {cm[1,1]} (correctly identified churners)")
print(f"True Negatives:  {cm[0,0]} (correctly identified non-churners)")
print(f"False Positives: {cm[0,1]} (wrongly flagged as churners)")
print(f"False Negatives: {cm[1,0]} (missed churners)")

In [ ]:
# ROC Curve & Precision-Recall Curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC
fpr, tpr, _ = roc_curve(y_test, best['y_prob'])
axes[0].plot(fpr, tpr, 'b-', linewidth=2, label=f"AUC = {best['roc']:.4f}")
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[0].fill_between(fpr, tpr, alpha=0.1)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve', fontweight='bold')
axes[0].legend(loc='lower right', fontsize=12)

# Precision-Recall
precision, recall, _ = precision_recall_curve(y_test, best['y_prob'])
axes[1].plot(recall, precision, 'r-', linewidth=2)
axes[1].fill_between(recall, precision, alpha=0.1, color='red')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve', fontweight='bold')

plt.tight_layout()
plt.show()

## 10. Feature Importance

In [ ]:
# Get feature names from preprocessor
feature_names = numerical_features.copy()
ohe = preprocessor.named_transformers_['cat']
cat_names = ohe.get_feature_names_out(categorical_features).tolist()
feature_names.extend(cat_names)

# Try to get feature importance from the best model
model_for_importance = best['model']
if hasattr(model_for_importance, 'feature_importances_'):
    importances = model_for_importance.feature_importances_
elif hasattr(model_for_importance, 'estimators_'):
    # For voting classifier, average importances
    imps = []
    for _, est in model_for_importance.estimators:
        if hasattr(est, 'feature_importances_'):
            imps.append(est.feature_importances_)
    importances = np.mean(imps, axis=0) if imps else None
else:
    importances = None

if importances is not None and len(feature_names) == len(importances):
    imp_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
    imp_df = imp_df.sort_values('Importance', ascending=True).tail(20)

    plt.figure(figsize=(10, 8))
    plt.barh(imp_df['Feature'], imp_df['Importance'], color='steelblue', edgecolor='black')
    plt.xlabel('Importance')
    plt.title('Top 20 Feature Importances', fontweight='bold', fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print("Feature importance not directly available for this model type.")

## 11. Save Model Artifacts

In [ ]:
# Save preprocessor
with open('preprocessor.pkl', 'wb') as f:
    pickle.dump(preprocessor, f)
print("Saved: preprocessor.pkl")

# Save best model
with open('best_model.pkl', 'wb') as f:
    pickle.dump(best['model'], f)
print("Saved: best_model.pkl")

# Save feature configuration
feature_config = {
    'categorical_features': categorical_features,
    'numerical_features': numerical_features,
    'service_cols': service_cols,
    'threshold': best['thresh'],
    'model_name': best['name'],
    'accuracy': best['acc'],
    'f1': best['f1'],
    'auc': best['roc'],
}
with open('feature_config.pkl', 'wb') as f:
    pickle.dump(feature_config, f)
print("Saved: feature_config.pkl")

print(f"\nModel: {best['name']}")
print(f"Accuracy: {best['acc']:.4f} | F1: {best['f1']:.4f} | AUC: {best['roc']:.4f}")

## 12. Test Prediction

In [ ]:
# Load saved artifacts
with open('best_model.pkl', 'rb') as f:
    loaded_model = pickle.load(f)
with open('preprocessor.pkl', 'rb') as f:
    loaded_preprocessor = pickle.load(f)
with open('feature_config.pkl', 'rb') as f:
    config = pickle.load(f)

# Example customer data
example = {
    'gender': 'Female', 'SeniorCitizen': 0, 'Partner': 'Yes',
    'Dependents': 'No', 'tenure': 1, 'PhoneService': 'Yes',
    'MultipleLines': 'Yes', 'InternetService': 'Fiber optic',
    'OnlineSecurity': 'No', 'OnlineBackup': 'No',
    'DeviceProtection': 'No', 'TechSupport': 'No',
    'StreamingTV': 'No', 'StreamingMovies': 'No',
    'Contract': 'Month-to-month', 'PaperlessBilling': 'Yes',
    'PaymentMethod': 'Electronic check',
    'MonthlyCharges': 70.35, 'TotalCharges': 70.35,
}

# Apply feature engineering (same as training)
input_df = pd.DataFrame([example])

# Simplify labels
input_df['MultipleLines'] = input_df['MultipleLines'].replace('No phone service', 'No')
for c in ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']:
    input_df[c] = input_df[c].replace('No internet service', 'No')

# Engineered features
input_df['AvgChargesPerMonth'] = input_df['TotalCharges'] / (input_df['tenure'] + 1)
input_df['TenureGroup'] = pd.cut(input_df['tenure'], bins=[-1, 12, 24, 48, 60, 72],
                                  labels=['0-12', '13-24', '25-48', '49-60', '61-72'])
input_df['ChargesRatio'] = input_df['TotalCharges'] / (input_df['MonthlyCharges'] * (input_df['tenure'] + 1) + 1)
svc = ['PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
       'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
input_df['NumServices'] = sum((input_df[c].isin(['Yes', 'DSL', 'Fiber optic'])).astype(int) for c in svc)
input_df['HasInternet'] = (input_df['InternetService'] != 'No').astype(int)
input_df['HasBundle'] = ((input_df['PhoneService'] == 'Yes') & (input_df['InternetService'] != 'No')).astype(int)
input_df['ChargesPerService'] = input_df['MonthlyCharges'] / (input_df['NumServices'] + 1)
input_df['IsNewCustomer'] = (input_df['tenure'] <= 12).astype(int)
input_df['NumSecurityFeatures'] = sum((input_df[c] == 'Yes').astype(int) for c in ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport'])
input_df['NumStreamingFeatures'] = sum((input_df[c] == 'Yes').astype(int) for c in ['StreamingTV', 'StreamingMovies'])

# Predict
X_new = loaded_preprocessor.transform(input_df)
prob = loaded_model.predict_proba(X_new)[0, 1]
threshold = config['threshold']
prediction = 'Churn' if prob >= threshold else 'No Churn'

print(f"Prediction:  {prediction}")
print(f"Probability: {prob:.4f} (threshold: {threshold:.3f})")
print(f"Model:       {config['model_name']}")
print(f"Accuracy:    {config['accuracy']:.4f}")